In [10]:
from dotenv import load_dotenv
import os
load_dotenv()

GROQ_API_KEY=os.getenv("GROQ_API_KEY")

In [11]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [12]:
from langchain_groq import ChatGroq
llm = ChatGroq(
    model="llama-3.3-70b-versatile"
)

In [13]:
response = llm.invoke(
    "Explain Large Language Models in one sentence."
)

print(response.content)

Large Language Models (LLMs) are artificial intelligence systems trained on vast amounts of text data to generate human-like language, allowing them to understand, process, and produce coherent and contextually relevant text based on the input they receive.


In [24]:
from youtube_transcript_api import YouTubeTranscriptApi
ytt_api = YouTubeTranscriptApi()

def get_transcript(video_id):
    transcript =ytt_api.fetch(video_id)
    text = " ".join(
        chunk.text
        for chunk in transcript
    )
    return text


In [27]:
video_id = "aqsrBQXZxWs"

transcript = ytt_api.fetch(video_id)

first_chunk = transcript[0]

print(type(first_chunk))
print(first_chunk)
print(dir(first_chunk))

<class 'youtube_transcript_api._transcripts.FetchedTranscriptSnippet'>
FetchedTranscriptSnippet(text='[music]', start=4.584, duration=6.256)
['__annotations__', '__class__', '__dataclass_fields__', '__dataclass_params__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__match_args__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'duration', 'start', 'text']


In [28]:
video_id ="aqsrBQXZxWs"
transcript=get_transcript(video_id)

document=Document(
    page_content=transcript,
    metadata={
        "source" :"youtube"
    }
)

In [29]:
splitter=RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200
)
docs=splitter.split_documents([document])

In [31]:
prompt=ChatPromptTemplate.from_template(
    """
    You are an expert note maker for students.
    Summarize the following transript chunk
    
    Transcript:
    {text}
     """
)

In [32]:
parser=StrOutputParser()

chain=(prompt | llm |parser)

In [33]:
chunk_summaries=[]
for doc in docs:
    summary=chain.invoke(
        {
            "text":doc.page_content
        }
    )
    chunk_summaries.append(summary)

In [35]:
combined_summary = "\n".join(
    chunk_summaries
)

In [36]:
final_prompt = ChatPromptTemplate.from_template(
    """
    Create a coherent final summary from the following notes.

    Notes:
    {notes}
    """
)

final_chain = (
    final_prompt
    | llm
    | parser
)

final_summary = final_chain.invoke(
    {
        "notes": combined_summary
    }
)

print(final_summary)

The song lyrics revolve around a complex and intense emotional landscape, characterized by a mix of romantic and dark themes. The speaker is deeply drawn to someone, despite acknowledging the potential harm and toxicity of the relationship, and craves physical and emotional connection. However, this desire is accompanied by feelings of guilt, shame, and desperation, as the speaker struggles with the danger and addiction of the relationship. Ultimately, the song appears to explore the destructive tendencies that can arise from intense passion, with the speaker seemingly unable to resist the allure of the toxic relationship, succumbing to the darkness and chaos that it embodies.
